# Testbench inventory, artifacts, and report coverage

This notebook is rendered in the Typst report via `@preview/callisto`. It is **executed offline** (for example with `jupyter nbconvert --execute --inplace`) and the resulting outputs are cached inside the `.ipynb` file.

In [1]:
from __future__ import annotations

import ast
import datetime as _dt
import re
import tomllib
import xml.etree.ElementTree as _ET
from dataclasses import dataclass
from pathlib import Path

from IPython.display import Markdown, display


def _find_repo_root(start: Path) -> Path:
    for cand in (start, *start.parents):
        if (cand / "testbench" / "targets.toml").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root (missing testbench/targets.toml)")


REPO_ROOT = _find_repo_root(Path.cwd())
TARGETS_PATH = REPO_ROOT / "testbench" / "targets.toml"
SIM_BUILD_ROOT = REPO_ROOT / "testbench" / "sim_build"
REPORT_SECTIONS = REPO_ROOT / "docs" / "report" / "sections"

assert TARGETS_PATH.exists(), f"Expected {TARGETS_PATH} after locating repo root."


def _human_bytes(num: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num < 1024 or unit == "TB":
            return f"{num:.1f} {unit}" if unit != "B" else f"{num} {unit}"
        num /= 1024
    return f"{num:.1f} TB"


def _md_table(rows: list[dict], *, columns: list[str]) -> str:
    def esc(val: object) -> str:
        s = "" if val is None else str(val)
        s = s.replace("|", r"\|")
        s = s.replace("\n", "<br>")
        return s

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(esc(r.get(c, "")) for c in columns) + " |" for r in rows]
    return "\n".join([header, sep, *body])


@dataclass(frozen=True, slots=True)
class TargetRow:
    key: str
    toplevel: str
    test_module: str
    description: str
    waves_enabled: bool
    build_dir: Path
    test_file: Path


@dataclass(frozen=True, slots=True)
class CaseResult:
    name: str
    status: str
    time_s: float
    sim_time_ns: float
    error_type: str | None = None
    error_msg: str | None = None


@dataclass(frozen=True, slots=True)
class TestDef:
    name: str
    lineno: int
    doc: str


@dataclass(frozen=True, slots=True)
class ParsedTestModule:
    module_doc: str
    tests: list[TestDef]


def _parse_results(results_xml: Path) -> list[CaseResult]:
    tree = _ET.parse(results_xml)
    root = tree.getroot()

    out: list[CaseResult] = []
    for tc in root.findall(".//testcase"):
        name = tc.attrib.get("name", "<unknown>")
        time_s = float(tc.attrib.get("time", "0") or "0")
        sim_time_ns = float(tc.attrib.get("sim_time_ns", "0") or "0")

        failure = tc.find("failure")
        error = tc.find("error")
        skipped = tc.find("skipped")

        if failure is not None:
            out.append(
                CaseResult(
                    name=name,
                    status="FAIL",
                    time_s=time_s,
                    sim_time_ns=sim_time_ns,
                    error_type=failure.attrib.get("error_type"),
                    error_msg=failure.attrib.get("error_msg"),
                )
            )
        elif error is not None:
            out.append(
                CaseResult(
                    name=name,
                    status="ERROR",
                    time_s=time_s,
                    sim_time_ns=sim_time_ns,
                    error_type=error.attrib.get("error_type"),
                    error_msg=error.attrib.get("error_msg"),
                )
            )
        elif skipped is not None:
            out.append(CaseResult(name=name, status="SKIP", time_s=time_s, sim_time_ns=sim_time_ns))
        else:
            out.append(CaseResult(name=name, status="PASS", time_s=time_s, sim_time_ns=sim_time_ns))

    return out


def _test_file_for_module(test_module: str) -> Path:
    # targets.toml uses import paths relative to testbench/ (e.g., tests.test_axi_rgb_to_grayscale)
    return REPO_ROOT / "testbench" / (test_module.replace(".", "/") + ".py")


def _is_cocotb_test_decorator(deco: ast.expr) -> bool:
    # Accept @cocotb.test, @cocotb.test(...), and @test / @test(...)
    func = deco.func if isinstance(deco, ast.Call) else deco

    if isinstance(func, ast.Attribute):
        return isinstance(func.value, ast.Name) and func.value.id == "cocotb" and func.attr == "test"

    if isinstance(func, ast.Name):
        return func.id == "test"

    return False


def _parse_test_module(path: Path) -> ParsedTestModule:
    module_ast = ast.parse(path.read_text(), filename=str(path))
    module_doc = ast.get_docstring(module_ast) or ""

    tests: list[TestDef] = []
    for node in module_ast.body:
        if isinstance(node, ast.AsyncFunctionDef) and any(
            _is_cocotb_test_decorator(d) for d in node.decorator_list
        ):
            tests.append(
                TestDef(
                    name=node.name,
                    lineno=int(getattr(node, "lineno", 0) or 0),
                    doc=(ast.get_docstring(node) or "").strip(),
                )
            )

    return ParsedTestModule(module_doc=module_doc.strip(), tests=sorted(tests, key=lambda t: t.lineno))


def _load_targets() -> list[TargetRow]:
    doc = tomllib.loads(TARGETS_PATH.read_text())
    defaults = doc.get("defaults", {})
    default_waves = bool(defaults.get("waves", False))

    targets: dict = doc.get("targets", {})
    rows: list[TargetRow] = []

    for key, cfg in targets.items():
        toplevel = cfg["toplevel"]
        test_module = cfg["test_module"]
        desc = cfg.get("description", "")
        waves_enabled = bool(cfg.get("waves", default_waves))
        mod_dir = SIM_BUILD_ROOT / test_module.split(".")[-1]
        build_dir = mod_dir / f"{key}_{toplevel}" / "build"
        rows.append(
            TargetRow(
                key=key,
                toplevel=toplevel,
                test_module=test_module,
                description=desc,
                waves_enabled=waves_enabled,
                build_dir=build_dir,
                test_file=_test_file_for_module(test_module),
            )
        )

    return rows


def _scan_report_coverage() -> tuple[set[str], set[str]]:
    text = ""
    for p in sorted(REPORT_SECTIONS.glob("*.typ")):
        text += p.read_text() + "\n"

    # 1) test-file mentions in academic_test_table(...) blocks
    test_files = set(re.findall(r'test_file:\s*"([^"]+)"', text))

    # 2) included waveform snapshot images
    ghw_imgs = set(re.findall(r'figures/ghw/([^"]+)', text))

    return test_files, ghw_imgs


def _ghw_snapshot_present(*, target_key: str, toplevel: str, report_ghw_imgs: set[str]) -> bool:
    # Conservative heuristic with one explicit alias for historical naming.
    aliases = {target_key, toplevel}
    if target_key == "axi_rgb_to_grayscale":
        aliases |= {"rgb2gray", "rgb_to_grayscale", "axi_rgb2gray"}

    return any(any(a in img for a in aliases) for img in report_ghw_imgs)


now = _dt.datetime.now().strftime("%Y-%m-%d %H:%M")
display(Markdown(f"_Executed:_ **{now}**  |  _Repo root:_ `{REPO_ROOT}`"))

_Executed:_ **2026-02-24 10:53**  |  _Repo root:_ `/Users/jd/Desktop/repos/realtime-image-processing`

In [2]:
targets = sorted(_load_targets(), key=lambda t: t.key)
report_test_files, report_ghw_imgs = _scan_report_coverage()

target_rows: list[dict] = []
fail_rows: list[dict] = []

# One row per testcase across all targets (used for overview + cross-target filtering).
case_index_rows: list[dict] = []

for t in targets:
    results_xml = t.build_dir / "results.xml"
    if not results_xml.exists():
        raise FileNotFoundError(f"Missing results.xml for target '{t.key}': {results_xml}")

    if not t.test_file.exists():
        raise FileNotFoundError(f"Missing test module file for target '{t.key}': {t.test_file}")

    cases = _parse_results(results_xml)
    parsed_mod = _parse_test_module(t.test_file)
    defs_by_name = {td.name: td for td in parsed_mod.tests}

    total = len(cases)
    pass_n = sum(1 for c in cases if c.status == "PASS")
    fail_n = sum(1 for c in cases if c.status in {"FAIL", "ERROR"})
    skip_n = sum(1 for c in cases if c.status == "SKIP")

    wall_s = sum(c.time_s for c in cases)
    sim_ns = sum(c.sim_time_ns for c in cases)

    ghw_files = sorted(t.build_dir.glob("*.ghw"))
    ghw_size = None
    ghw_name = None
    if ghw_files:
        ghw_name = ghw_files[0].name
        ghw_size = _human_bytes(ghw_files[0].stat().st_size)

    png_files = sorted(t.build_dir.glob("*.png"))

    test_file_basename = t.test_module.split(".")[-1] + ".py"
    has_component_table = test_file_basename in report_test_files

    has_ghw_snapshot = _ghw_snapshot_present(
        target_key=t.key,
        toplevel=t.toplevel,
        report_ghw_imgs=report_ghw_imgs,
    )

    target_rows.append(
        {
            "target": t.key,
            "dut": t.toplevel,
            "tests": f"{pass_n}/{total}",
            "fails": fail_n,
            "skips": skip_n,
            "wall_s": f"{wall_s:.2f}",
            "sim_ms": f"{sim_ns/1e6:.2f}",
            "waves": "on" if t.waves_enabled else "off",
            "ghw": (ghw_name + (f" ({ghw_size})" if ghw_size else "")) if ghw_name else "-",
            "png": str(len(png_files)),
            "component_overview_table": "yes" if has_component_table else "no",
            "wave_snapshot_in_report": "yes" if has_ghw_snapshot else "no",
        }
    )

    for c in cases:
        td = defs_by_name.get(c.name)
        doc = (td.doc if td else "").strip()
        if not doc:
            # Fallback: readable phrase from function name, without repeating the module-level description.
            doc = c.name.removeprefix("test_").replace("_", " ")

        case_index_rows.append(
            {
                "target": t.key,
                "testcase": c.name,
                "status": c.status,
                "wall_s": f"{c.time_s:.3f}",
                "sim_us": f"{c.sim_time_ns/1e3:.1f}",
                "description": doc,
            }
        )

        if c.status in {"FAIL", "ERROR"}:
            fail_rows.append(
                {
                    "target": t.key,
                    "testcase": c.name,
                    "status": c.status,
                    "error": (c.error_type or "") + ("" if not c.error_msg else f": {c.error_msg}"),
                }
            )

summary = {
    "targets": len(targets),
    "testcases_total": len(case_index_rows),
    "testcases_failed": sum(1 for r in case_index_rows if r["status"] in {"FAIL", "ERROR"}),
    "waves_targets": sum(1 for t in targets if t.waves_enabled),
}

display(
    Markdown(
        f"**Totals:** {summary['targets']} targets, {summary['testcases_total']} testcases, {summary['testcases_failed']} failing.  \\\n"
        f"**Waves enabled:** {summary['waves_targets']} targets (per `targets.toml`)."
    )
)

cols = [
    "target",
    "dut",
    "tests",
    "fails",
    "skips",
    "wall_s",
    "sim_ms",
    "waves",
    "ghw",
    "png",
    "component_overview_table",
    "wave_snapshot_in_report",
]
display(Markdown("## Per-target results and artifacts"))
display(Markdown(_md_table(target_rows, columns=cols)))

if fail_rows:
    display(Markdown("## Failing testcases (details)"))
    display(Markdown(_md_table(fail_rows, columns=["target", "testcase", "status", "error"])))
else:
    display(Markdown("## Failing testcases"))
    display(Markdown("No failing testcases in the cached results."))

# Test overview index (self-contained, directly extracted from test sources/results.xml)
display(Markdown("## Test overview (extracted from source + joined with results.xml)"))
display(
    Markdown(
        "The table below provides a self-contained overview of every cocotb testcase across all registered targets. "
        "If a testcase has no docstring, the description falls back to a readable version of the function name."
    )
)
display(
    Markdown(
        _md_table(
            case_index_rows,
            columns=["target", "testcase", "description", "status", "wall_s", "sim_us"],
        )
    )
)

missing_component_tables = [r for r in target_rows if r["component_overview_table"] == "no"]
missing_wave_snaps = [
    r
    for r in target_rows
    if r["waves"] == "on" and r["ghw"] != "-" and r["wave_snapshot_in_report"] == "no"
]

# Clarify what is missing *outside* this notebook.
display(Markdown("## Report coverage gaps (component-local documentation)"))
display(
    Markdown(
        "This notebook itself provides full regression results + an extracted testcase index. "
        "The remaining gaps below refer to *component-local* documentation inside the report sections "
        "(for example `academic_test_table` blocks and curated waveform snapshots)."
    )
)

display(
    Markdown(
        f"- Missing **component-local test overview table** (`academic_test_table`): **{len(missing_component_tables)}** targets\n"
        f"- Missing **waveform snapshot figure** in report (wave exists but not embedded): **{len(missing_wave_snaps)}** targets"
    )
)

if missing_component_tables:
    display(Markdown("### Targets missing a component-local test overview table"))
    display(
        Markdown(
            _md_table(
                missing_component_tables,
                columns=["target", "dut", "tests", "fails", "ghw"],
            )
        )
    )

if missing_wave_snaps:
    display(Markdown("### Targets missing a waveform snapshot figure"))
    display(Markdown(_md_table(missing_wave_snaps, columns=["target", "dut", "ghw"])))

**Totals:** 16 targets, 37 testcases, 7 failing.  \
**Waves enabled:** 11 targets (per `targets.toml`).

## Per-target results and artifacts

| target | dut | tests | fails | skips | wall_s | sim_ms | waves | ghw | png | component_overview_table | wave_snapshot_in_report |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| axi_blurr_window_module | axi_blurrwindowmodule | 1/3 | 2 | 0 | 63.58 | 7.87 | off | - | 0 | no | no |
| axi_frame_compositor | axi_framecompositor | 7/7 | 0 | 0 | 206.24 | 28.38 | on | axi_framecompositor.ghw (583.4 MB) | 0 | yes | no |
| axi_gray_blurr_sobel_overlay_pipeline | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | 0 | 5095.88 | 5.24 | off | - | 2 | no | no |
| axi_gray_blurr_sobel_overlay_pipeline_downscaled | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | 0 | 6.76 | 0.08 | off | - | 1 | no | no |
| axi_gray_blurr_sobel_overlay_pipeline_synth_fsm_axi | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | 0 | 0.16 | 0.01 | off | - | 0 | no | no |
| axi_rgb_to_grayscale | axi_rgbtograyscale | 3/3 | 0 | 0 | 12.15 | 2.62 | on | axi_rgbtograyscale.ghw (26.8 MB) | 1 | no | yes |
| axi_sobel_filter | axi_sobelfilter | 1/2 | 1 | 0 | 9.99 | 2.62 | on | axi_sobelfilter.ghw (26.9 MB) | 0 | no | no |
| axi_sobel_window_module | axi_sobelwindowmodule | 1/3 | 2 | 0 | 61.36 | 7.87 | off | - | 0 | no | no |
| example_passthrough | example_passthrough | 2/2 | 0 | 0 | 8.38 | 2.62 | on | example_passthrough.ghw (11.4 MB) | 0 | no | no |
| frame_compositor_core | framecompositor | 1/1 | 0 | 0 | 0.00 | 0.00 | on | framecompositor.ghw (1.0 KB) | 0 | yes | no |
| shift_ram_chain | shiftramchain | 2/2 | 0 | 0 | 0.02 | 0.00 | on | shiftramchain.ghw (202.7 KB) | 0 | yes | yes |
| test_click_detector | clickdetector | 1/1 | 0 | 0 | 0.00 | 0.00 | on | clickdetector.ghw (1.6 KB) | 0 | no | no |
| test_debounced_click_detector | debouncedclickdetector | 1/1 | 0 | 0 | 0.00 | 0.00 | on | debouncedclickdetector.ghw (5.8 KB) | 0 | no | no |
| test_debouncer | debouncer | 1/1 | 0 | 0 | 0.00 | 0.00 | on | debouncer.ghw (1.9 KB) | 0 | no | no |
| test_example | example_passthrough | 2/2 | 0 | 0 | 6.74 | 2.62 | on | example_passthrough.ghw (11.4 MB) | 0 | no | no |
| window_generator | window_generator | 1/3 | 2 | 0 | 0.02 | 0.00 | on | window_generator.ghw (16.7 KB) | 0 | no | no |

## Failing testcases (details)

| target | testcase | status | error |
| --- | --- | --- | --- |
| axi_blurr_window_module | test_axi_blurr_window_module_simple_image | FAIL | AssertionError: First mismatch at (x=2, y=0): expected=2, received=3 |
| axi_blurr_window_module | test_axi_blurr_window_module_lenna_end_to_end | FAIL | AssertionError: First mismatch at (x=0, y=0): expected=87, received=155 |
| axi_sobel_filter | test_axi_sobel_filter_lenna_end_to_end | FAIL | AssertionError: First mismatch at (x=422, y=1): expected=0, received=255 |
| axi_sobel_window_module | test_axi_sobel_window_module_simple_image | FAIL | AssertionError: First mismatch at (x=48, y=0): expected=[255, 255, 255], received=[0, 0, 0] |
| axi_sobel_window_module | test_axi_sobel_window_module_lenna_end_to_end | FAIL | AssertionError: First mismatch at (x=0, y=0): expected=[255, 255, 255], received=[0, 0, 0] |
| window_generator | test_axi_rgb_to_window_without_pressure | FAIL | AssertionError: Window mismatch(es) at [0, 1, 2, 3, 4, 5, 9, 10, 14, 15] |
| window_generator | test_axi_rgb_to_window_with_pressure | FAIL | AssertionError: Window mismatch(es) at [0, 1, 2, 3, 4, 5, 9, 10, 14, 15] |

...

- Missing **component-local test overview table** (`academic_test_table`): **13** targets
- Missing **waveform snapshot figure** in report (wave exists but not embedded): **9** targets

### Targets missing a component-local test overview table

| target | dut | tests | fails | ghw |
| --- | --- | --- | --- | --- |
| axi_blurr_window_module | axi_blurrwindowmodule | 1/3 | 2 | - |
| axi_gray_blurr_sobel_overlay_pipeline | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | - |
| axi_gray_blurr_sobel_overlay_pipeline_downscaled | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | - |
| axi_gray_blurr_sobel_overlay_pipeline_synth_fsm_axi | axi_rgbgrayblurrsobeloverlaypipeline | 2/2 | 0 | - |
| axi_rgb_to_grayscale | axi_rgbtograyscale | 3/3 | 0 | axi_rgbtograyscale.ghw (26.8 MB) |
| axi_sobel_filter | axi_sobelfilter | 1/2 | 1 | axi_sobelfilter.ghw (26.9 MB) |
| axi_sobel_window_module | axi_sobelwindowmodule | 1/3 | 2 | - |
| example_passthrough | example_passthrough | 2/2 | 0 | example_passthrough.ghw (11.4 MB) |
| test_click_detector | clickdetector | 1/1 | 0 | clickdetector.ghw (1.6 KB) |
| test_debounced_click_detector | debouncedclickdetector | 1/1 | 0 | debouncedclickdetector.ghw (5.8 KB) |
| test_debouncer | debouncer | 1/1 | 0 | debouncer.ghw (1.9 KB) |
| test_example | example_passthrough | 2/2 | 0 | example_passthrough.ghw (11.4 MB) |
| window_generator | window_generator | 1/3 | 2 | window_generator.ghw (16.7 KB) |

### Targets missing a waveform snapshot figure

| target | dut | ghw |
| --- | --- | --- |
| axi_frame_compositor | axi_framecompositor | axi_framecompositor.ghw (583.4 MB) |
| axi_sobel_filter | axi_sobelfilter | axi_sobelfilter.ghw (26.9 MB) |
| example_passthrough | example_passthrough | example_passthrough.ghw (11.4 MB) |
| frame_compositor_core | framecompositor | framecompositor.ghw (1.0 KB) |
| test_click_detector | clickdetector | clickdetector.ghw (1.6 KB) |
| test_debounced_click_detector | debouncedclickdetector | debouncedclickdetector.ghw (5.8 KB) |
| test_debouncer | debouncer | debouncer.ghw (1.9 KB) |
| test_example | example_passthrough | example_passthrough.ghw (11.4 MB) |
| window_generator | window_generator | window_generator.ghw (16.7 KB) |